# 04_ Blur & jpeg arrtifact
___

Nó thêm vào trong metadata các trường dữ liệu: `blur_scroe`,`jpeg_score`, `quality_flag`

In [ ]:
import cv2
import numpy as np

def laplacian_variance(image):
    gray = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2GRAY
    )
    return cv2.Laplacian(
        gray,
        cv2.CV_64F
    ).var()

## JPEG artifact

Có thể dùng blocking artifact / DCT-related statistics làm indicator. Một heuristic đơn giản:

In [ ]:
def jpeg_blockiness_score(image):
    gray = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2GRAY
    )
    h, w = gray.shape
    vertical = []
    horizontal = []

    for x in range(8, w, 8):
        diff = np.abs(
            gray[:, x].astype(float)
            -
            gray[:, x - 1].astype(float)
        )
        vertical.append(diff.mean())
    for y in range(8, h, 8):
        diff = np.abs(
            gray[y, :].astype(float)
            -
            gray[y - 1, :].astype(float)
        )
        horizontal.append(diff.mean())
    score = (
        np.mean(vertical)
        +
        np.mean(horizontal)
    ) / 2
    return score

## Generate metadata

In [ ]:
results = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    path = DATASET_ROOT / row["path"]
    image = cv2.imread(str(path))
    if image is None:
        continue
    blur = laplacian_variance(image)
    jpeg = jpeg_blockiness_score(image)
    results.append({
        "image_id": row["image_id"],
        "class_id": row["class_id"],
        "split": row["split"],
        "blur_score": blur,
        "jpeg_artifact_score": jpeg
    })
quality_df = pd.DataFrame(results)